In [75]:
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from histpy import Histogram

from scoords import SpacecraftFrame

from astropy.time import Time
import astropy.units as u

SED_KEV_TO_ERG = u.keV.to(u.erg)
KEV_TO_MEV = u.keV.to(u.MeV)
from astropy.coordinates import SkyCoord, Galactic

import numpy as np
import matplotlib.pyplot as plt

from threeML import *
from threeML.io.package_data import get_path_of_data_file
from threeML.io.logging import silence_console_log
from astromodels import Parameter
from threeML.minimizer.minimization import CannotComputeCovariance

from jupyterthemes import jtplot
jtplot.style(context="talk", fscale=1, ticks=True, grid=False)
set_threeML_style()
silence_warnings()

from scipy.integrate import quad

import matplotlib.ticker as mticker

from pathlib import Path

import os

%matplotlib inline

In [76]:
data_path = Path("/Users/parshadkp/Software/COSI_Data/")

In [77]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import draw_energy_hist_mev, open_spacecraft_history, scale_spacecraft_livetime

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
source_coord = SkyCoord(l=172.104, b=-51.934, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

full_sc_orientation = open_spacecraft_history(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    full_sc_orientation,
    fov_cut,
    earth_occ=False,
)
sc_orientation = full_sc_orientation.apply_gti(source_gti)

print(f"NGC 1068 FOV cut: {fov_cut.to_value(u.deg):.0f} deg")
print(f"Selected livetime: {sc_orientation.cumulative_livetime().to_value(u.s):,.1f} s")


WARNING RuntimeWarning: invalid value encountered in subtract


WARNING RuntimeWarning: invalid value encountered in subtract



NGC 1068 FOV cut: 60 deg
Selected livetime: 2,546,115.0 s


In [78]:
dr = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"

In [79]:
multiplier_1068 = 8
exposure_1068 = multiplier_1068 * 3

# Scale count histograms and response livetime together; keep source flux intrinsic.
sc_orientation = scale_spacecraft_livetime(sc_orientation, multiplier_1068)

## Cutoff Power Law (Thermal)

#### NGC 1068 (128 keV)

In [92]:
K_inj = 3.5e-1 / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 128. * u.keV
index_inj = -1.9

spectrum_inj_ec128 = Cutoff_powerlaw()

spectrum_inj_ec128.K.value = K_inj.value
spectrum_inj_ec128.piv.value = piv_inj.value
spectrum_inj_ec128.xc.value = xc_inj.value
spectrum_inj_ec128.index.value = index_inj

spectrum_inj_ec128.K.unit = K_inj.unit
spectrum_inj_ec128.piv.unit = piv_inj.unit
spectrum_inj_ec128.xc.unit = xc_inj.unit

## Cutoff Powerlaw (Non-thermal)

#### NGC 1068 (128 keV) Non-thermal cutoff

In [93]:
# K_inj = 0.1*spectrum_inj_ec128.evaluate_at(1000) / u.cm / u.cm / u.s / u.keV
# piv_inj = 1000. * u.keV

def cutoff_powerlaw_k_at_pivot(shape, pivot_value):
    return float(shape.K.value * (pivot_value / shape.piv.value) ** shape.index.value)

K_inj = 0.10*spectrum_inj_ec128.evaluate_at(1) / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
index_inj = -1.85
cutoff_inj = 3750. * u.keV

spectrum_inj_ec128_PL = Cutoff_powerlaw()

spectrum_inj_ec128_PL.K.value = K_inj.value
spectrum_inj_ec128_PL.piv.value = piv_inj.value
spectrum_inj_ec128_PL.index.value = index_inj
spectrum_inj_ec128_PL.xc.value = cutoff_inj.value

spectrum_inj_ec128_PL.K.unit = K_inj.unit
spectrum_inj_ec128_PL.piv.unit = piv_inj.unit
spectrum_inj_ec128_PL.xc.unit = cutoff_inj.unit

spectrum_inj_ec128_total = spectrum_inj_ec128 + spectrum_inj_ec128_PL

fit_pivot_keV = 200.0
thermal_k_at_fit_pivot = cutoff_powerlaw_k_at_pivot(spectrum_inj_ec128, fit_pivot_keV)
tail_k_at_fit_pivot = cutoff_powerlaw_k_at_pivot(spectrum_inj_ec128_PL, fit_pivot_keV)
linking_ratio_ec128 = tail_k_at_fit_pivot / thermal_k_at_fit_pivot

print(spectrum_inj_ec128.evaluate_at(fit_pivot_keV))
print(spectrum_inj_ec128_PL.evaluate_at(fit_pivot_keV))
print("="*40)
print("Linking K ratio", linking_ratio_ec128)
print("Flux ratio at 200 keV", spectrum_inj_ec128_PL.evaluate_at(fit_pivot_keV)/spectrum_inj_ec128.evaluate_at(fit_pivot_keV))
print("="*40)
flux_th, _ = quad(spectrum_inj_ec128.evaluate_at, 200.0, 5000.0)
flux_nth, _ = quad(spectrum_inj_ec128_PL.evaluate_at, 200.0, 5000.0)
print("Non-thermal flux: ", flux_nth)
print("Total flux: ", flux_th + flux_nth)
print("Thermal - Non-thermal Flux Ratio: ", flux_th/flux_nth)
print("Non-thermal percentage: ", flux_nth/(flux_th + flux_nth))

3.1154868651488162e-06
1.8222470438467002e-06
Linking K ratio 0.12931787935796996
Flux ratio at 200 keV 0.5848996072591877
Non-thermal flux:  0.00035317801024010873
Total flux:  0.0005583406505018627
Thermal - Non-thermal Flux Ratio:  0.5809043437395034
Non-thermal percentage:  0.6325493404835485


# Spectral Fitting

In [94]:
source_file = data_path / "AGN_Data/GammaRay/Paper_Models/NGC1068_ec_128_-1.85_DC4_COSI_cpl_cpl_60_fovCut.hdf5"
background_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Background/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_NGC1068_60deg_fov_cut.hdf5")

for required_file in (source_file, background_file):
    if not required_file.exists():
        raise FileNotFoundError(required_file)

NGC1068_ec128 = Histogram.open(source_file) * multiplier_1068
bkg = Histogram.open(background_file) * multiplier_1068

# Collapse the background time axis and match the source histogram metadata.
bkg = bkg.project("Em", "Phi", "PsiChi")
NGC1068_ec128.axes["Em"].axis_scale = bkg.axes["Em"].axis_scale
NGC1068_ec128 = NGC1068_ec128.to(
    unit=bkg.unit,
    update=False,
)

NGC1068_ec128_bkg = NGC1068_ec128 + bkg

print(f"Loaded source: {source_file.name}")
print(f"Loaded background: {background_file.name}")


Loaded source: NGC1068_ec_128_-1.85_DC4_COSI_cpl_cpl_60_fovCut.hdf5
Loaded background: Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_NGC1068_60deg_fov_cut.hdf5


In [ ]:
FONT_SIZE = 25
plt.rcParams['agg.path.chunksize'] = 10000

plt.rcParams.update({'font.size': FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams.update({
    'font.weight': '550',
    'axes.titleweight': '550',
    'axes.labelweight': '550'
})

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)

ax.xaxis.set_tick_params(which='major', size=12, width=1.5, direction='in', top='on', pad=8, labelsize=FONT_SIZE)
ax.xaxis.set_tick_params(which='minor', size=6, width=1.5, direction='in', top='on', pad=8)
ax.yaxis.set_tick_params(which='major', size=12, width=1.5, direction='in', right='on', pad=8, labelsize=FONT_SIZE)
ax.yaxis.set_tick_params(which='minor', size=6, width=1.5, direction='in', right='on', pad=8)

ax.spines['right'].set_visible(True)
ax.spines['top'].set_visible(True)

draw_energy_hist_mev(NGC1068_ec128, ax, label="Simulated ($E_c$ = 128 keV)", color="red", linestyle="dotted")
draw_energy_hist_mev(bkg, ax, label="DC4 Time-Cut Background", color="grey")
# draw_energy_hist_mev(full_bkg, ax, label="DC4 Full Background", color="black", linestyle="dashdot")
# draw_energy_hist_mev(NGC1068_ec128_bkg, ax, label="NGC 1068 Model ($E_c$ = 128 keV) + Background", linestyle='dashdot', color="red")

text = f'NGC 1068 (Cpl + Cpl) \n{exposure_1068}-months'
ax.text(
    0.02, 0.98,
    text,
    transform=ax.transAxes,
    ha='left', va='top',
    fontsize=FONT_SIZE,
    fontweight='550'
)

ax.set_yscale("log")
ax.set_xscale("log")

ax.set_ylabel("Counts", fontsize=FONT_SIZE)
ax.set_ylim(1e-2, 5e9)

# ax.xaxis.set_major_locator(mticker.FixedLocator([0.2, 1, 5]))
# ax.xaxis.set_major_formatter(
#     mticker.FixedFormatter(["0.2", "1.0", "5.0"])
# )

ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.legend(fontsize=FONT_SIZE, loc='upper right', frameon=False)

save_path = f"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC/Plots/NGC1068_Cpl_Cpl_Counts_128_{exposure_1068}Months.pdf"
# plt.savefig(save_path)

## Perform spectral fit

Set background parameter, which is used to fit the amplitude of the background, and instantiate the COSI 3ML plugin

In [95]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
bkg_par = Parameter("background_cosi",                                         # background parameter
                     1,                                                        # initial value of parameter
                     min_value=0,                                              # minimum value of parameter
                     max_value=5,                                              # maximum value of parameter
                     delta=0.05,                                               # initial step used by fitting engine
                     desc="Background parameter for cosi")

cosi = COSIPlugin("cosi",                                                        # COSI 3ML plugin
                 dr = dr,                                                      # detector response
                 data = NGC1068_ec128_bkg.project('Em', 'Phi', 'PsiChi'),   # data (source+background)
                 bkg = bkg.project('Em', 'Phi', 'PsiChi'),         # background model 
                 sc_orientation = sc_orientation,                              # spacecraft orientation
                 nuisance_param = bkg_par,                                     # background parameter
                 earth_occ = True)                                             # Option to account for Earth occultation

### Powerlaw with energy cutoff fit

In [96]:
l=172.104
b=-51.934

# Cpl component used in the Cpl+Cpl and Cpl-only model comparisons.
K = thermal_k_at_fit_pivot / u.cm / u.cm / u.s / u.keV
piv = fit_pivot_keV * u.keV
xc = 400. * u.keV
index = -1.9

spectrum_cpl = Cutoff_powerlaw()

spectrum_cpl.K.value = K.value
spectrum_cpl.piv.value = piv.value
spectrum_cpl.xc.value = xc.value
spectrum_cpl.index.value = index
spectrum_cpl.index.fix = True

spectrum_cpl.K.min_value = 1e-8
spectrum_cpl.K.max_value = 1e-2
spectrum_cpl.xc.min_value = 10
spectrum_cpl.xc.max_value = 2000
# spectrum_cpl.index.min_value = -3
# spectrum_cpl.index.max_value = 1

spectrum_cpl.K.unit = K.unit
spectrum_cpl.piv.unit = piv.unit
spectrum_cpl.xc.unit = xc.unit



## Thermal + Non-thermal Fit

In [97]:
l=172.104
b=-51.934

# Give it some harsher initial guesses
K = tail_k_at_fit_pivot / u.cm / u.cm / u.s / u.keV
piv = fit_pivot_keV * u.keV
xc = 1200. * u.keV
index = -1.85

spectrum = Cutoff_powerlaw()

spectrum.K.value = K.value
spectrum.piv.value = piv.value
spectrum.xc.value = xc.value
spectrum.index.value = index
# spectrum.index.fix = True

# Harsher Parameters
spectrum.K.min_value = 1e-8
spectrum.K.max_value = 1e-2
spectrum.xc.min_value = 1000 # keep these relatively the same
spectrum.xc.max_value = 10000 # change to 1000
spectrum.index.min_value = -3 # change to -3.5 to sample lower, larger values (-5, 5)
spectrum.index.max_value = 1

# spectrum_cpl.K.delta = 0.05
# spectrum_cpl.xc.delta = 10
# spectrum_cpl.index.delta = 0.15

spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit
spectrum.xc.unit = xc.unit



In [98]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
# Keep both spectral components in one point source so the response cache
# tracks every component as linked or independently fitted parameters change.
source1 = PointSource(
    "source1",
    l=l,
    b=b,
    spectral_shape=spectrum_cpl + spectrum,
)
model = Model(source1)
cosi.set_model(model)


def copy_parameter_settings(reference_parameter, target_parameter):
    target_parameter.value = reference_parameter.value
    target_parameter.fix = reference_parameter.fix

    if reference_parameter.min_value is not None:
        target_parameter.min_value = reference_parameter.min_value
    if reference_parameter.max_value is not None:
        target_parameter.max_value = reference_parameter.max_value
    if reference_parameter.delta is not None:
        target_parameter.delta = reference_parameter.delta
    if reference_parameter.unit is not None:
        target_parameter.unit = reference_parameter.unit


def make_cpl_only_source(name, reference_source):
    reference_shape = reference_source.spectrum.main.shape.functions[0]
    cpl_shape = Cutoff_powerlaw()

    for parameter_name in ("K", "piv", "xc", "index"):
        copy_parameter_settings(
            getattr(reference_shape, parameter_name),
            getattr(cpl_shape, parameter_name),
        )
    return PointSource(
        name,
        l=l,
        b=b,
        spectral_shape=cpl_shape,
    )


def make_pl_only_source(name):
    pl_shape = Powerlaw()
    pl_shape.K.value = (1e-5 / u.cm / u.cm / u.s / u.keV).value
    pl_shape.piv.value = (200. * u.keV).value
    pl_shape.index.value = -3

    pl_shape.K.min_value = 1e-8
    pl_shape.K.max_value = 1e-2
    pl_shape.index.min_value = -5
    pl_shape.index.max_value = 1
    pl_shape.index.delta = 0.25

    pl_shape.K.unit = u.cm ** -2 * u.s ** -1 * u.keV ** -1
    pl_shape.piv.unit = u.keV

    return PointSource(
        name,
        l=l,
        b=b,
        spectral_shape=pl_shape,
    )


def make_cosi_plugin(name, data_hist, nuisance_name):
    nuisance_parameter = Parameter(
        nuisance_name,
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc=f"Background parameter for {name}",
    )

    return COSIPlugin(
        name,
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=nuisance_parameter,
        earth_occ=True,
    )


source1_cpl_only = make_cpl_only_source("source1_cpl_only", source1)
source1_pl_only = make_pl_only_source("source1_pl_only")

model_cpl_only = Model(source1_cpl_only)
model_pl_only = Model(source1_pl_only)

cosi_cpl_only = make_cosi_plugin(
    "cosi_cpl_only",
    NGC1068_ec128_bkg,
    "background_cosi_cpl_only",
)
cosi_pl_only = make_cosi_plugin(
    "cosi_pl_only",
    NGC1068_ec128_bkg,
    "background_cosi_pl_only",
)

cosi_cpl_only.set_model(model_cpl_only)
cosi_pl_only.set_model(model_pl_only)


### Joint Fit

In [ ]:
# bat_ec128.use_effective_area_correction(0.02, 1.8)
# plugins = DataList(bat_ec128, cosi)

# bat_ec1000.use_effective_area_correction(0.02, 1.8)
# plugins_ec1000 = DataList(bat_ec1000, cosi_ec1000)

### Only COSI Fit

In [99]:
## Only COSI fit
plugins = DataList(cosi)
plugins_cpl_only = DataList(cosi_cpl_only)
plugins_pl_only = DataList(cosi_pl_only)

In [102]:
ratio_ec128 = linking_ratio_ec128
link_function = Line(a=0.0, b=ratio_ec128)   # tail K = a + b * source1.K
link_function.a.fix = True
link_function.b.min_value = 0.0  # Require a non-negative non-thermal normalization.

model.link(
    model.source1.spectrum.main.composite.K_2,  # dependent tail normalization
    model.source1.spectrum.main.composite.K_1,  # independent thermal normalization
    link_function,
)

like = JointLikelihood(model, plugins, verbose=False)
like_cpl_only = JointLikelihood(model_cpl_only, plugins_cpl_only, verbose=False)
like_pl_only = JointLikelihood(model_pl_only, plugins_pl_only, verbose=False)
result = like.fit()
result_cpl_only = like_cpl_only.fit()
result_pl_only = like_pl_only.fit()

14:48:25 INFO      set the minimizer to minuit                                             ]8;id=541722;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=43839;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=699904;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=487930;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to minuit                                             ]8;id=484834;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=62388;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
source1.spectrum.main.composite.K_1,(1.49 -0.19 +0.22) x 10^-5,1 / (keV s cm2)
source1.spectrum.main.composite.xc_1,(1.28 -0.12 +0.13) x 10^2,keV
source1.spectrum.main.composite.K_2.Line.b,(1.29 +/- 0.20) x 10^-1,
source1.spectrum.main.composite.index_2,-1.84 +/- 0.07,
source1.spectrum.main.composite.xc_2,(3.7 -1.2 +1.7) x 10^3,keV
background_cosi,(2.48983 +/- 0.00011) x 10,Hz


...

1.00,0.07,-0.69
0.07,1.00,-0.27
-0.69,-0.27,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_pl_only,-3794621063.1489964
total,-3794621063.1489964


Values of statistical measures:

,statistical measures
AIC,-7589242120.297889
BIC,-7589242089.255276


### Cpl + Cpl vs Cpl vs Pl comparison

In [103]:
from agn_cosi_fit_utils import COSIPlugin, EnergyRangeCOSIPlugin, make_cosi_background_parameter
def make_null_likelihood(data_hist, label):
    bkg_par_null = Parameter(
        f"background_cosi_null_{label}",
        1,
        min_value=0,
        max_value=5,
        delta=0.05,
        desc="Background parameter for the null COSI fit",
    )

    cosi_null = COSIPlugin(
        f"cosi_null_{label}",
        dr=dr,
        data=data_hist.project("Em", "Phi", "PsiChi"),
        bkg=bkg.project("Em", "Phi", "PsiChi"),
        sc_orientation=sc_orientation,
        nuisance_param=bkg_par_null,
        earth_occ=True,
    )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "source_null",
        l=l,
        b=b,
        spectral_shape=spectrum_null,
    )

    model_null = Model(source_null)
    cosi_null.set_model(model_null)

    plugins_null = DataList(cosi_null)
    like_null = JointLikelihood(model_null, plugins_null, verbose=False)
    like_null.fit()

    return like_null


def get_likelihood_statistic(joint_likelihood):
    statistic_frame = joint_likelihood.results.get_statistic_frame()
    statistic_series = statistic_frame["-log(likelihood)"]

    if "total" in statistic_series.index:
        return float(statistic_series.loc["total"])

    return float(statistic_series.sum())


def detection_ts(null_likelihood, source_likelihood):
    return 2.0 * (
        get_likelihood_statistic(null_likelihood)
        - get_likelihood_statistic(source_likelihood)
    )


def model_delta_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


like_null_128 = make_null_likelihood(NGC1068_ec128_bkg, "ec128")

TS_cpl_cpl_128 = detection_ts(like_null_128, like)
TS_cpl_128 = detection_ts(like_null_128, like_cpl_only)
TS_pl_128 = detection_ts(like_null_128, like_pl_only)
TS_cpl_cpl_vs_cpl_128 = model_delta_ts(like_cpl_only, like)
TS_cpl_cpl_vs_pl_128 = model_delta_ts(like_pl_only, like)
# TS_cpl_vs_pl_128 = model_delta_ts(like_pl_only, like_cpl_only)

fit_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": "NGC 1068 Ec=128 keV",
            "TS_Cpl_plus_Cpl": TS_cpl_cpl_128,
            "TS_Cpl": TS_cpl_128,
            "TS_Pl": TS_pl_128,
            "Delta_TS_Cpl_plus_Cpl_vs_Cpl": TS_cpl_cpl_vs_cpl_128,
            "Delta_TS_Cpl_plus_Cpl_vs_Pl": TS_cpl_cpl_vs_pl_128,
            # "Delta_TS_Cpl_vs_Pl": TS_cpl_vs_pl_128,
            "Sigma_Cpl_plus_Cpl": np.sqrt(max(TS_cpl_cpl_128, 0.0)),
            "Sigma_Cpl": np.sqrt(max(TS_cpl_128, 0.0)),
            "Sigma_Pl": np.sqrt(max(TS_pl_128, 0.0)),
            "Sigma_added_Cpl_plus_Cpl_vs_Cpl": np.sqrt(max(TS_cpl_cpl_vs_cpl_128, 0.0)),
            "Sigma_added_Cpl_plus_Cpl_vs_Pl": np.sqrt(max(TS_cpl_cpl_vs_pl_128, 0.0)),
        },
    ]
)

display(fit_ts_comparison)

for _, row in fit_ts_comparison.iterrows():
    print(row["spectrum"])
    print(f"  Cpl + Cpl TS: {row['TS_Cpl_plus_Cpl']:.3f} ({row['Sigma_Cpl_plus_Cpl']:.2f} sigma)")
    print(f"  Cpl TS: {row['TS_Cpl']:.3f} ({row['Sigma_Cpl']:.2f} sigma)")
    print(f"  Pl TS: {row['TS_Pl']:.3f} ({row['Sigma_Pl']:.2f} sigma)")
    print(f"  Cpl + Cpl vs Cpl dTS: {row['Delta_TS_Cpl_plus_Cpl_vs_Cpl']:.3f}")
    print(f"  Cpl + Cpl vs Pl dTS: {row['Delta_TS_Cpl_plus_Cpl_vs_Pl']:.3f}")
    # print(f"  Cpl vs Pl dTS: {row['Delta_TS_Cpl_vs_Pl']:.3f}")

# Backward-compatible alias used by the plotting cells below.
TS = TS_cpl_cpl_128

14:48:29 INFO      set the minimizer to minuit                                             ]8;id=601068;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=158578;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
background_cosi_null_ec128,(2.49601 +/- 0.00011) x 10,Hz


Correlation matrix:

1.00


...

,-log(likelihood)
cosi_null_ec128,-3794618810.4322004
total,-3794618810.4322004


Values of statistical measures:

,statistical measures
AIC,-7589237618.864384
BIC,-7589237608.516829


,spectrum,TS_Cpl_plus_Cpl,TS_Cpl,TS_Pl,Delta_TS_Cpl_plus_Cpl_vs_Cpl,Delta_TS_Cpl_plus_Cpl_vs_Pl,Sigma_Cpl_plus_Cpl,Sigma_Cpl,Sigma_Pl,Sigma_added_Cpl_plus_Cpl_vs_Cpl,Sigma_added_Cpl_plus_Cpl_vs_Pl
0,NGC 1068 Ec=128 keV,4520.12396,4319.945342,4505.433592,200.178617,14.690368,67.231867,65.726291,67.122527,14.148449,3.832802


NGC 1068 Ec=128 keV
  Cpl + Cpl TS: 4520.124 (67.23 sigma)
  Cpl TS: 4319.945 (65.73 sigma)
  Pl TS: 4505.434 (67.12 sigma)
  Cpl + Cpl vs Cpl dTS: 200.179
  Cpl + Cpl vs Pl dTS: 14.690
